In [ ]:
# Quietly load the files
.output <- source("./Functions/lecospectR.R", echo = FALSE, verbose = FALSE)

In [ ]:
get_filename <- function(
    bandwidth, 
    count, 
    is_train = TRUE, 
    base_path = "Data/v2/") {
    if (is_train) {
        train_test_string <- "train"
    } else {
        train_test_string <- "test"
    }

    return(
        paste0(
            base_path,
            train_test_string,
            "_",
            bandwidth,
            "nm_",
            count,
            ".csv"
        )
    )
}

In [ ]:
seeds <- c(
    #6265,
    #4041,
    #3621,
    #942,
    #3143,
    #1764,
    #1378,
    #3964,
    #4270,
    #5542,
    #1816,
    #2833,
    4024,
    3031,
    6389,
    1368,
    4900,
    4075,
    6232,
    7118,
    7590,
    7928,
    2725,
    2422,
    7475,
    3857,
    1652,
    2041,
    9017,
    6447
)

In [ ]:
data_raw <- read.csv(file = "Data/Ground_Validation/PFT_image_spectra/PFT_Image_SpectralLib_Clean.csv")

In [ ]:
indices <- get_vegetation_indices(df = data_raw, ml_model = NULL) 


In [ ]:
bands <- resample_df(df = data_raw)
colnames(bands)

In [ ]:
num_cols_initial <- ncol(data_raw)
num_cols_bands <- ncol(bands)



In [ ]:
df <- cbind(bands, indices)

In [ ]:
colnames(df)

In [ ]:
num_per_pft <- 300
train_test_data <- create_patch_balanced_sample(
    df,
    test_count = 20,
    train_count = NULL,
    verbose = FALSE
    )

split_2_train <- create_patch_balanced_sample(
    train_test_data$remainder,
    test_count = num_per_pft,
    train_count = 1000,
    verbose = FALSE
    )

In [ ]:
exclude_vars <- c('X', "UID", "ScanNum", "sample_name", "PFT", "FncGrp1", "Site")

target_variable <- "FncGrp1"

In [ ]:
calculate_model_metrics <- function(
    model, 
    test_data, 
    test_labels, 
    seed,
    n = 4,
    model_dir = "./",
    manifest_path = "./seed_and_size.csv") {
    # create predictions (ranger)

        model_id <- uuid::UUIDgenerate()

        model_dir_full <- paste0(
                model_dir,
                model_id,
                "/"
            )

        if(!dir.exists(model_dir)){
            dir.create(model_dir)
        }

        dir.create(
            model_dir_full
        )


        model_predictions <- predict(
            model,
            test_data
        )$prediction %>% as.factor()

        confusion_matrix <- caret::confusionMatrix(
            model_predictions %>% to_fg0(),
            test_labels %>% as.factor() %>% to_fg0() %>% add_forb(),
            mode = "everything"
        )

        acc <- as.list(confusion_matrix$overall)$Accuracy
        print(paste0("Model Accuracy: ", acc))

        validate_model(
            model,
            save_directory = model_dir_full
        )

        aggregated_results <- aggregate_results(model_dir_full)
        r2 <- calculate_validation_r2(aggregated_results)
        rpd <- calculate_rpd(aggregated_results)

        print(r2)

        save(model, file = paste0(model_dir_full, "model.rda"))

        print(model_dir)
        plt <- plot_by_pft(
            aggregated_results,
            save_path = paste0(model_dir_full, "aggregate.html"),
            open = FALSE,
            image_path = NULL,
            aggregation = 0
        )
        
        add_model_to_manifest(
            model_id = model_id,
            model_type = "Random Forest",
            bandwidth = 5,
            max_count = 300,
            preprocessing = paste0(
                "none"
            ),
            max_correlation = "none",
            weight = "balanced",
            hyperparam1 = n,
            # oob_error = model$prediction.error,
            accuracy = acc,
            r2 = r2,
            rpd = rpd,
            seed = seed,
            logpath = manifest_path
        )
}

In [ ]:
for(seed in seeds){
    set.seed(seed)

    # split the data based on the seed
    num_per_pft <- 300
    train_test_data <- create_patch_balanced_sample(
        df,
        test_count = 20,
        train_count = NULL,
        verbose = FALSE
        )

    split_2_train <- create_patch_balanced_sample(
        train_test_data$remainder,
        test_count = num_per_pft,
        train_count = 1000,
        verbose = FALSE
        )
    labels <- split_2_train$selection[,target_variable] %>% as.factor()
    test_labels <- train_test_data$selection[,target_variable] %>% as.factor()
    
    include_vars <- setdiff(
        colnames(split_2_train$selection),
        colnames(data_raw)
        )

    train_data <- impute_spectra(
        #clip_outliers(
            subset(
                x = split_2_train$selection,
                select = include_vars
                )
            )
        #)
    test_data <- impute_spectra(
        #clip_outliers(
            subset(
                train_test_data$selection,
                select = include_vars
                )
            )
        #)
    #if (("Forb" %in% levels(labels)) && !("Forb" %in% levels(test_labels))) {
    #    levels(test_labels) <- c(levels(test_labels), "Forb")
    #}
        
    small_model <- ranger::ranger(
        num.trees = 4,
        replace = TRUE,
        classification = TRUE,
        # alpha = a,
        case.weights = targets_to_weights(labels),
        x = train_data,
        y = labels
    )

    # reset seed
    set.seed(seed)
    big_model <- ranger::ranger(
        num.trees = 1000,
        replace = TRUE,
        classification = TRUE,
        # alpha = a,
        case.weights = targets_to_weights(labels),
        x = impute_spectra(train_data),
        y = labels
    )

    # and again for better reproducability
    set.seed(seed)
    calculate_model_metrics(
        small_model,
        train_test_data$selection[, include_vars],
        train_test_data$selection[, target_variable] %>% as.factor(),
        seed,
        n = 4,
        model_dir = "./temp/"
    )

    calculate_model_metrics(
        big_model,
        train_test_data$selection[, include_vars],
        train_test_data$selection[, target_variable] %>% as.factor(),
        seed,
        n = 1000,
        model_dir = "./temp/"
    )

    print(paste0("Iteration completed for seed ", seed))
}

In [18]:
results <- read.csv("./seed_and_size.csv")

head(results)

,maxCount,bandwidth,maxCorrelation,preprocessing,weight,hyperparam1,hyperparam2,accuracy,r2,rpd,seed,model_id
,<int>,<int>,<chr>,<chr>,<chr>,<int>,<lgl>,<dbl>,<dbl>,<dbl>,<int>,<chr>
1,300,5,none,none,balanced,4,NA,0.83125,0.3603392,1.051130,6265,179e38c2-416f-489a-86b2-2f1eb7c47425
2,300,5,none,none,balanced,1000,NA,0.86250,0.3342386,1.350129,6265,916eaee3-1c9c-4ae9-b152-890461b8e44a
3,300,5,none,none,balanced,4,NA,0.81250,0.3211105,1.073748,4041,b50f6b23-ad71-4231-aa51-e39e9363e9b5
4,300,5,none,none,balanced,1000,NA,0.90625,0.3128352,1.333134,4041,32812298-6adb-4402-ab24-8c5106017dbe
5,300,5,none,none,balanced,4,NA,0.81875,0.4142811,1.166157,3621,eac3817e-e5f4-4b5d-9ba5-94efb8c6467d
6,300,5,none,none,balanced,1000,NA,0.87500,0.3748756,1.329950,3621,665a10b2-7d29-46f4-8ea6-5cdbc21854b7


In [19]:
small_results <- results[results$hyperparam1 == 4,] %>% as.data.frame()
large_results <- results[results$hyperparam1 == 1000,] %>% as.data.frame()
head(small_results)

,maxCount,bandwidth,maxCorrelation,preprocessing,weight,hyperparam1,hyperparam2,accuracy,r2,rpd,seed,model_id
,<int>,<int>,<chr>,<chr>,<chr>,<int>,<lgl>,<dbl>,<dbl>,<dbl>,<int>,<chr>
1,300,5,none,none,balanced,4,NA,0.83125,0.3603392,1.051130,6265,179e38c2-416f-489a-86b2-2f1eb7c47425
3,300,5,none,none,balanced,4,NA,0.81250,0.3211105,1.073748,4041,b50f6b23-ad71-4231-aa51-e39e9363e9b5
5,300,5,none,none,balanced,4,NA,0.81875,0.4142811,1.166157,3621,eac3817e-e5f4-4b5d-9ba5-94efb8c6467d
7,300,5,none,none,balanced,4,NA,0.79375,0.3389215,1.141278,942,fe3d3725-6cad-4b9d-8dce-32cda9e9495a
9,300,5,none,none,balanced,4,NA,0.85625,0.4971012,1.036625,3143,a4e814d5-c425-4f7b-a719-56f2b293b048
11,300,5,none,none,balanced,4,NA,0.83125,0.3603392,1.051130,6265,04eccb5d-8048-4efb-97c7-c31cd35a73d1


In [20]:
print(small_results$accuracy)

 [1] 0.83125 0.81250 0.81875 0.79375 0.85625 0.83125 0.81250 0.81875 0.79375
[10] 0.85625 0.81250 0.82500 0.75000 0.79375 0.83125 0.81250


In [21]:
is.atomic(small_results$accuracy)

[1] TRUE

In [22]:
s_acc_m <- mean(small_results$accuracy)
s_acc_v <- var(small_results$accuracy, y = small_results$accuracy, na.rm = TRUE)
s_r2_m <- mean(small_results$r2)
s_r2_v <- var(small_results$r2, y = small_results$r2, na.rm = TRUE)
s_rpd_m <- mean(small_results$rpd)
s_rpd_v <- var(small_results$rpd, y = small_results$rpd)

l_acc_v <- var(large_results$accuracy, y = large_results$accuracy)
l_acc_m <- mean(large_results$accuracy %>% as.numeric())
l_r2_m <- mean(large_results$r2 %>% as.numeric())
l_r2_v <- var(large_results$r2, y = large_results$r2)
l_rpd_m <- mean(large_results$rpd)
l_rpd_v <- var(large_results$rpd,y = large_results$rpd)

print(
        paste("           Accuracy           Variance           r-squared           Variance            RPD           Variance")
)
        print(paste("Large", l_acc_m, l_acc_v, l_r2_m, l_r2_v, l_rpd_m, l_rpd_v, sep = "  "))
        print(paste("Small", s_acc_m, s_acc_v, s_r2_m, s_r2_v, s_rpd_m, s_rpd_v, sep = "  "))
    

[1] "           Accuracy           Variance           r-squared           Variance            RPD           Variance"
[1] "Large  0.88203125  0.000488932291666667  0.329590427502085  0.000835114376457031  1.34778571202534  0.000977985894329167"
[1] "Small  0.815625  0.000661458333333333  0.383110249980788  0.00352967384797665  1.08914677831256  0.00284910106901266"
